# 04 Statistical Analysis

In this notebook, we perform statistical tests to validate the relationships observed in EDA, focusing specifically on our core business problem: testing for congestion and location bias in severity assignment.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with {df.shape[0]} rows.")

Loaded dataset with 95607 rows.


## 1. Congestion Bias Analysis (Rush Hour vs Severity)

**Hypothesis:** If high severity is merely a proxy for congestion, we should see a statistically significant association between Rush Hour and Severity level.

**Test:** Chi-Square Test of Independence

In [2]:
contingency_table = pd.crosstab(df['Is_Rush_Hour'], df['Severity'])
display(contingency_table)

chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-Square Statistic: {chi2:.2f}")
print(f"P-value: {p:.4e}")

Severity,1,2,3,4
Is_Rush_Hour,,,,
False,4472,16811,18838,11669
True,5477,15836,16150,6354


Chi-Square Statistic: 1248.33
P-value: 2.3915e-270


**Business Interpretation:**
A very small p-value indicates a significant relationship between Rush Hour and Severity. However, given large sample sizes, almost any difference is statistically significant. We must look at the cross-tabulation in EDA to see the practical effect size.

## 2. Duration Difference across Severity Levels

Severity is defined as the impact on traffic (delay). Let's verify if `Duration_min` significantly differs across severity levels using the Kruskal-Wallis H Test (since duration is highly skewed and non-normal).

In [3]:
durations_by_sev = [df[df['Severity'] == s]['Duration_min'].dropna() for s in sorted(df['Severity'].unique())]

h_stat, p_val = stats.kruskal(*durations_by_sev)
print(f"Kruskal-Wallis H Statistic: {h_stat:.2f}")
print(f"P-value: {p_val:.4e}")

print("\nMedian Duration (minutes) by Severity:")
print(df.groupby('Severity')['Duration_min'].median())

Kruskal-Wallis H Statistic: 11941.02
P-value: 0.0000e+00

Median Duration (minutes) by Severity:
Severity
1     44.716667
2     59.083333
3     44.516667
4    131.366667
Name: Duration_min, dtype: float64


**Business Interpretation:**
If the test is significant and median durations align strongly with severity, it validates the dataset's definition of Severity as a measure of traffic delay. If Severity 4 has a significantly longer duration, it confirms it represents major blockages.

## 3. Weather Conditions vs Severity

Does bad weather lead to higher severity delays?

In [4]:
# Top 5 weather conditions for simpler test
top_weather = df['Weather_Condition'].value_counts().nlargest(5).index
weather_subset = df[df['Weather_Condition'].isin(top_weather)]

weather_contingency = pd.crosstab(weather_subset['Weather_Condition'], weather_subset['Severity'])
chi2_w, p_w, _, _ = stats.chi2_contingency(weather_contingency)

print(f"Chi-Square Statistic (Weather vs Severity): {chi2_w:.2f}")
print(f"P-value: {p_w:.4e}")

# Show row proportions
display(weather_contingency.div(weather_contingency.sum(axis=1), axis=0).round(3) * 100)

Chi-Square Statistic (Weather vs Severity): 3618.04
P-value: 0.0000e+00


Severity,1,2,3,4
Weather_Condition,,,,
Clear,0.3,32.7,47.0,19.9
Cloudy,13.3,35.4,30.4,20.8
Fair,17.6,36.6,27.1,18.7
Mostly Cloudy,12.0,32.6,38.8,16.7
Partly Cloudy,11.6,33.8,39.1,15.5


**Business Interpretation:**
Identifies if certain weather conditions (like snow or fog) systematically result in more severe delays. This helps logistics companies route around specific weather fronts.

## 4. Location Bias: State vs Severity

Are some states more prone to reporting high severity?

In [5]:
state_severity = pd.crosstab(df['State'], df['Severity'], normalize='index') * 100
state_severity['High_Sev_Pct'] = state_severity[3] + state_severity[4]
top_bias_states = state_severity.sort_values('High_Sev_Pct', ascending=False).head(10)
print("Top 10 States with highest percentage of Severity 3+4 accidents:")
display(top_bias_states[['High_Sev_Pct']])

Top 10 States with highest percentage of Severity 3+4 accidents:


Severity,High_Sev_Pct
State,
WY,89.473684
SD,87.500000
AR,82.770270
WI,81.686047
MO,80.916031
GA,80.431034
KS,80.076628
CT,79.083665
IA,78.297872


**Business Interpretation:**
If certain states overwhelmingly report Severity 3 or 4 while others report Severity 2, it suggests inconsistencies in how different traffic authorities classify severity. Models trained on this data might learn state-level reporting biases rather than true accident risk.